## 에어컨 테이블 전처리

In [1]:
from pathlib import Path
import pandas as pd
import re

# =========================
# 경로 설정
# =========================
input_path = Path(r"C:/project_skn/project4/4th_project/products/data/preprocessing/AC/lge_ac_all_products.csv")
output_path = Path("ProductAC.csv")

# =========================
# 변환 함수
# =========================
def blank_if_na(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def clean_int(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "")
    nums = re.findall(r"\d+", text)

    if not nums:
        return pd.NA

    return int(nums[0])


def clean_float(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "")
    nums = re.findall(r"\d+(?:\.\d+)?", text)

    if not nums:
        return pd.NA

    return float(nums[0])


def clean_proficiency(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).strip()

    if "비대상" in text:
        return pd.NA

    # 냉방 2등급 / 난방 4등급 -> 냉방 기준
    cool_match = re.search(r"냉방\s*(\d+)", text)
    if cool_match:
        return int(cool_match.group(1))

    nums = re.findall(r"\d+", text)
    if not nums:
        return pd.NA

    return int(nums[0])


def clean_max_value(value):
    """
    예: 1,830 / 310 -> 1830
    예: 6,500 / 1,600 -> 6500
    """
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "")
    nums = re.findall(r"\d+", text)

    if not nums:
        return pd.NA

    return max(map(int, nums))


def split_voltage_hertz(value):
    """
    예: 220, 60 -> voltage=220, hertz=60
    값이 하나거나 없으면 둘 다 공란
    """
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA, pd.NA

    text = str(value).replace(",", " ")
    nums = re.findall(r"\d+", text)

    if len(nums) < 2:
        return pd.NA, pd.NA

    return int(nums[0]), int(nums[1])


def split_size(value):
    """
    예: 870 x 650 x 330
    예: "380 x 1,915 x 295"
    """
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA, pd.NA, pd.NA

    text = str(value).replace(",", "").replace('"', "").strip()
    parts = re.split(r"\s*[xX×]\s*", text)

    if len(parts) < 3:
        return pd.NA, pd.NA, pd.NA

    width = clean_int(parts[0])
    height = clean_int(parts[1])
    depth = clean_int(parts[2])

    return width, height, depth


def clean_coverage(value):
    """
    예: 71.5(52.8+18.7) -> 71.5
    """
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).strip()
    before_parenthesis = text.split("(")[0]

    return clean_float(before_parenthesis)


def clean_dehumid(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).strip()

    if "있음" in text:
        return 1
    if "없음" in text:
        return 0

    return pd.NA


# =========================
# 원본 읽기
# =========================
df = pd.read_csv(input_path)

# =========================
# 크기 분리
# =========================
in_size = df["실내기 크기(mm)"].apply(split_size)
out_size = df["실외기 크기(mm)"].apply(split_size)

df["in_width(mm)"] = in_size.apply(lambda x: x[0])
df["in_height(mm)"] = in_size.apply(lambda x: x[1])
df["in_depth(mm)"] = in_size.apply(lambda x: x[2])

df["out_width(mm)"] = out_size.apply(lambda x: x[0])
df["out_height(mm)"] = out_size.apply(lambda x: x[1])
df["out_depth(mm)"] = out_size.apply(lambda x: x[2])

# =========================
# 전압 / Hz 분리
# =========================
voltage_hertz = df["정격전압(V)"].apply(split_voltage_hertz)

df["voltage(V)"] = voltage_hertz.apply(lambda x: x[0])
df["hertz(Hz)"] = voltage_hertz.apply(lambda x: x[1])

# =========================
# 새 DataFrame 구성
# =========================
new_df = pd.DataFrame({
    "product_code": [f"ACT{i:03d}" for i in range(len(df))],
    "name": df["제품명"].apply(blank_if_na),
    "img_link": df["이미지 링크"].apply(blank_if_na),
    "price": df["가격(원)"].apply(clean_int),
    "proficiency_level": df["에너지 소비효율등급"].apply(clean_proficiency),
    "power_consum(W)": df["냉방 소비전력(W)"].apply(clean_max_value),
    "cool_cap(W)": df["냉방능력(W)"].apply(clean_max_value),
    "voltage(V)": df["voltage(V)"],
    "hertz(Hz)": df["hertz(Hz)"],
    "in_width(mm)": df["in_width(mm)"],
    "in_height(mm)": df["in_height(mm)"],
    "in_depth(mm)": df["in_depth(mm)"],
    "out_width(mm)": df["out_width(mm)"],
    "out_height(mm)": df["out_height(mm)"],
    "out_depth(mm)": df["out_depth(mm)"],
    "coverage(㎡)": df["냉방면적(㎡)"].apply(clean_coverage),
    "wind_speed(level)": df["바람세기"].apply(clean_int),
    "dehumid": df["제습"].apply(clean_dehumid),
    "color": df["색상"].apply(blank_if_na),
    "manual_link": df["제품사용설명서"].apply(blank_if_na)
})

# =========================
# 타입 지정
# =========================
str_cols = [
    "product_code", "name", "img_link", "color", "manual_link"
]

int_cols = [
    "price", "proficiency_level", "power_consum(W)", "cool_cap(W)",
    "voltage(V)", "hertz(Hz)",
    "in_width(mm)", "in_height(mm)", "in_depth(mm)",
    "out_width(mm)", "out_height(mm)", "out_depth(mm)",
    "wind_speed(level)", "dehumid"
]

for col in str_cols:
    new_df[col] = new_df[col].astype("string")

for col in int_cols:
    new_df[col] = new_df[col].astype("Int64")

new_df["coverage(㎡)"] = new_df["coverage(㎡)"].astype("Float64")

# =========================
# 저장
# =========================
new_df.to_csv(output_path, index=False, encoding="utf-8-sig", na_rep="")

print(new_df.head())
print(f"저장 완료: {output_path}")

  product_code                                   name  \
0       ACT000      LG 휘센 오브제컬렉션 타워II 에어컨 2in1 (4시리즈)   
1       ACT001  LG 휘센 AI 오브제컬렉션 뷰I 프로 에어컨 2in1 (6시리즈)   
2       ACT002      LG 휘센 오브제컬렉션 타워II 에어컨 2in1 (1시리즈)   
3       ACT003      LG 휘센 오브제컬렉션 타워II 에어컨 2in1 (1시리즈)   
4       ACT004      LG 휘센 오브제컬렉션 타워II 에어컨 2in1 (1시리즈)   

                                            img_link    price  \
0  https://www.lge.co.kr/kr/images/air-conditione...     <NA>   
1  https://www.lge.co.kr/kr/images/air-conditione...  4107000   
2  https://www.lge.co.kr/kr/images/air-conditione...     <NA>   
3  https://www.lge.co.kr/kr/images/air-conditione...     <NA>   
4  https://www.lge.co.kr/kr/images/air-conditione...     <NA>   

   proficiency_level  power_consum(W)  cool_cap(W)  voltage(V)  hertz(Hz)  \
0                  2             1830         6500         220         60   
1                  2             2100         7200         220         60   
2                  2             20